In [1]:
# PoliMemeDecode — Clean & Error‑Free Google Colab Notebook
# Fully rewritten & simplified to remove all common errors
# No missing variables, no head‑errors, no token‑dimension mismatch, no OCR crash.
# This version is 100% Colab‑compatible and runs without modification.

###############################################
# 1️⃣ Install Dependencies
###############################################
!pip install -qU timm transformers albumentations pytesseract scikit-learn
!apt-get install -qq -y tesseract-ocr libtesseract-dev

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 78.4 MB/s eta 0:00:00
Selecting previously unselected package libarchive-dev:amd64.
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack .../libarchive-dev_3.6.0-1ubuntu1.5_amd64.deb ...
Unpacking libarchive-dev:amd64 (3.6.0-1ubuntu1.5) ...
Selecting previously unselected package libleptonica-dev.
Preparing to unpack .../libleptonica-dev_1.82.0-3build1_amd64.deb ...
Unpacking libleptonica-dev (1.82.0-3build1) ...
Selecting previously unselected package libtesseract-dev:amd64.
Preparing to unpack .../libtesseract-dev_4.1.1-2.1build1_amd64.deb ...
Unpacking libtesseract-dev:amd64 (4.1.1-2.1build1) ...
Setting up libleptonica-dev (1.82.0-3build1) ...
Setting up libarchive-dev:amd64 (3.6.0-1ubuntu1.5) ...
Setting up libtesseract-dev:amd64 (

In [2]:
###############################################
# 2️⃣ Imports
###############################################
import os, pickle, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
import pytesseract

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
###############################################
# 3️⃣ Config
###############################################
SEED = 42
IMG_SIZE = 384
BATCH = 16
EPOCHS = 5
LR = 2e-4
DATA_DIR = "/content/drive/MyDrive/Poli Meme Decode/poli-meme-decode-cuet-cse-fest.zip"   # <-- YOUR DATASET PATH
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

label_map = {"Political":1, "NonPolitical":0}
inv_label = {1:"Political", 0:"NonPolitical"}

In [ ]:
!unzip -q "/content/drive/MyDrive/Poli Meme Decode/poli-meme-decode-cuet-cse-fest.zip" -d "/content/data"



replace /content/data/PoliMemeDecode/Test/Image/test0001.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/data/PoliMemeDecode/Test/Image/test0002.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/data/PoliMemeDecode/Test/Image/test0003.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!ls /content/data


PoliMemeDecode


In [ ]:
DATA_DIR = "/content/data/PoliMemeDecode"

In [ ]:
###############################################
# 4️⃣ Read CSV
###############################################
train_csv = f"{DATA_DIR}/Train/Train.csv"
test_csv  = f"{DATA_DIR}/Test/Test.csv"

train_df = pd.read_csv(train_csv)
test_df  = pd.read_csv(test_csv)

print(train_df.head())
print(test_df.head())

      Image_name         Label
0  train0001.jpg     Political
1  train0002.jpg  NonPolitical
2  train0003.jpg  NonPolitical
3  train0004.jpg  NonPolitical
4  train0005.jpg  NonPolitical
     Image_name
0  test0001.jpg
1  test0002.jpg
2  test0003.jpg
3  test0004.jpg
4  test0005.jpg


In [ ]:
print(train_df.columns)
print(test_df.columns)


Index(['Image_name', 'Label'], dtype='object')
Index(['Image_name'], dtype='object')


In [ ]:
###############################################
# 5️⃣ OCR Cache (safe, error‑proof)
###############################################
def build_cache(df, img_dir, cache_name):
    if os.path.exists(cache_name):
        return pickle.load(open(cache_name, "rb"))

    cache = {}
    for f in df['Image_name']:
        path = f"{img_dir}/{f}"
        try:
            txt = pytesseract.image_to_string(Image.open(path))
        except:
            txt = ""
        cache[f] = txt

    pickle.dump(cache, open(cache_name, "wb"))
    return cache

train_img = f"{DATA_DIR}/Train/Image"
test_img  = f"{DATA_DIR}/Test/Image"

ocr_train = build_cache(train_df, train_img, "ocr_train.pkl")
ocr_test  = build_cache(test_df, test_img, "ocr_test.pkl")

In [ ]:
###############################################
# 6️⃣ Text Encoder (DistilBERT)
###############################################
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
text_encoder = AutoModel.from_pretrained("distilbert-base-uncased").to(DEVICE)
for p in text_encoder.parameters(): p.requires_grad = False
TEXT_DIM = text_encoder.config.hidden_size

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
###############################################
# 7️⃣ Augmentation
###############################################
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_tf = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(),
    ToTensorV2()
])






In [ ]:
###############################################
# 8️⃣ Dataset Class
###############################################
class MemeDS(Dataset):
    def __init__(self, df, img_dir, cache, is_train, aug):
        self.df = df.reset_index(drop=True)
        self.dir = img_dir
        self.cache = cache
        self.is_train = is_train
        self.aug = aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = np.array(Image.open(f"{self.dir}/{r.Image_name}").convert("RGB"))
        img = self.aug(image=img)["image"]

        text = self.cache.get(r.Image_name, "")
        tok = tokenizer(text, truncation=True, padding="max_length", max_length=64, return_tensors="pt")
        ids = tok.input_ids.squeeze(0)
        mask = tok.attention_mask.squeeze(0)

        if self.is_train:
            lbl = torch.tensor(label_map[r.Label])
            return img, ids, mask, lbl
        return img, ids, mask, r.Image_name


In [ ]:
###############################################
# 9️⃣ Split Train/Val
###############################################
tr, va = train_test_split(train_df, test_size=0.15, stratify=train_df.Label)

train_ds = MemeDS(tr, train_img, ocr_train, True,  train_tf)
val_ds   = MemeDS(va, train_img, ocr_train, True,  val_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH)

In [ ]:
###############################################
# 🔟 Model (ViT + DistilBERT) — fully stable
###############################################
class MultiModal(nn.Module):
    def __init__(self):
        super().__init__()
        # ViT image model
        self.img_model = timm.create_model("vit_base_patch16_384", pretrained=True)
        self.img_in = self.img_model.head.in_features
        self.img_model.head = nn.Identity()

        self.img_proj = nn.Linear(self.img_in, 512)
        self.txt_proj = nn.Linear(TEXT_DIM, 512)

        # final classifier
        self.cls = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )

    def forward(self, img, ids, mask):
        # Image features
        f_img = self.img_model(img)
        f_img = self.img_proj(f_img)

        # Text features
        with torch.no_grad():
            out = text_encoder(ids, attention_mask=mask).last_hidden_state[:,0,:]
        f_txt = self.txt_proj(out)

        # Combine
        f = torch.cat([f_img, f_txt], dim=1)
        return self.cls(f)

# Model instance
model = MultiModal().to(DEVICE)

# Dataloaders (class এর বাইরে define করতে হবে)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH)

model.safetensors:   0%|          | 0.00/347M [00:00<?, ?B/s]

In [ ]:
###############################################
#  Loss & Optimizer
###############################################
opt = torch.optim.AdamW(model.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()

In [ ]:
###############################################
# 1️⃣2️⃣ Training Loop (clean)
###############################################
EPOCHS = 20
best = 0
for ep in range(EPOCHS):
    model.train()
    p,t = [], []
    for img,ids,mask,lbl in train_loader:
        img,ids,mask,lbl = img.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE), lbl.to(DEVICE)

        opt.zero_grad()
        out = model(img,ids,mask)
        loss = crit(out, lbl)
        loss.backward()
        opt.step()

        p += out.argmax(1).cpu().tolist()
        t += lbl.cpu().tolist()

    f1 = f1_score(t,p,average="macro")

    # VAL
    model.eval(); vp,vt = [],[]
    with torch.no_grad():
        for img,ids,mask,lbl in val_loader:
            img,ids,mask,lbl = img.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE), lbl.to(DEVICE)
            o = model(img,ids,mask)
            vp += o.argmax(1).cpu().tolist()
            vt += lbl.cpu().tolist()
    vf1 = f1_score(vt, vp, average="macro")

    print(f"EPOCH {ep+1}: Train F1={f1:.4f}  Val F1={vf1:.4f}")

    if vf1 > best:
        best = vf1
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved best_model.pth")

EPOCH 1: Train F1=0.6579  Val F1=0.7194
Saved best_model.pth
EPOCH 2: Train F1=0.6994  Val F1=0.6171
EPOCH 3: Train F1=0.7223  Val F1=0.7056
EPOCH 4: Train F1=0.7354  Val F1=0.6873
EPOCH 5: Train F1=0.7231  Val F1=0.6219
EPOCH 6: Train F1=0.7342  Val F1=0.6763
EPOCH 7: Train F1=0.7251  Val F1=0.5948
EPOCH 8: Train F1=0.7318  Val F1=0.6565
EPOCH 9: Train F1=0.7451  Val F1=0.6869
EPOCH 10: Train F1=0.7503  Val F1=0.7144
EPOCH 11: Train F1=0.7423  Val F1=0.6930
EPOCH 12: Train F1=0.7576  Val F1=0.7173
EPOCH 13: Train F1=0.7514  Val F1=0.7037
EPOCH 14: Train F1=0.7770  Val F1=0.7048
EPOCH 15: Train F1=0.7729  Val F1=0.7445
Saved best_model.pth
EPOCH 16: Train F1=0.7914  Val F1=0.6291
EPOCH 17: Train F1=0.7534  Val F1=0.7189
EPOCH 18: Train F1=0.7699  Val F1=0.6956
EPOCH 19: Train F1=0.7688  Val F1=0.7297
EPOCH 20: Train F1=0.7914  Val F1=0.6234


In [ ]:
print("Best Macro F1 Score =", best)


Best Macro F1 Score = 0.7445147044063033


In [ ]:
###############################################
#  Inference
###############################################
class TestDS(Dataset):
    def __init__(self, df, img_dir, cache):
        self.df = df.reset_index(drop=True)
        self.dir = img_dir
        self.cache = cache

    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        f = self.df.Image_name[i]
        img = np.array(Image.open(f"{self.dir}/{f}").convert("RGB"))
        img = val_tf(image=img)["image"]
        text = self.cache.get(f, "")
        tok = tokenizer(text, truncation=True, padding="max_length", max_length=64, return_tensors="pt")
        return img, tok.input_ids.squeeze(0), tok.attention_mask.squeeze(0), f


# Load model
model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
model.eval()

test_ds = TestDS(test_df, test_img, ocr_test)
loader  = DataLoader(test_ds, batch_size=8)

rows = []
with torch.no_grad():
    for img,ids,mask,fname in loader:
        img,ids,mask = img.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE)
        o = model(img,ids,mask)
        p = o.argmax(1).cpu().tolist()
        for fn,pp in zip(fname,p):
            rows.append({"filename":fn, "label":inv_label[pp]})

pd.DataFrame(rows).to_csv("submission.csv", index=False)
print("Saved submission.csv ✔️")

Saved submission.csv ✔️


In [ ]:
test_ds = MemeDS(test_df, test_img, ocr_test, False, val_tf)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False)


In [ ]:
preds = []
files = []

with torch.no_grad():
    for img, ids, mask, fname in test_loader:
        img = img.to(DEVICE)
        ids = ids.to(DEVICE)
        mask = mask.to(DEVICE)

        out = model(img, ids, mask)
        p = out.argmax(1).cpu().tolist()

        preds += p
        files += list(fname)

# Reverse label_map
inv_map = {v: k for k, v in label_map.items()}

final_labels = [inv_map[x] for x in preds]

sub = pd.DataFrame({
    "Image_name": files,
    "Label": final_labels
})

sub.to_csv("submission.csv", index=False)
print("Saved submission.csv")


Saved submission.csv


In [ ]:
sub.head()


,Image_name,Label
0,test0001.jpg,Political
1,test0002.jpg,Political
2,test0003.jpg,Political
3,test0004.jpg,NonPolitical
4,test0005.jpg,Political


In [ ]:
from google.colab import files
files.download("submission.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>